# Lesson 05: The Phi SLM Family — Textbook-Quality Data, Dense Reasoning & Edge Variants

In this companion notebook, you will:
1. Load an open-weight compact Phi / reasoning-capable SLM (or run a deterministic fallback on CPU) using Hugging Face `transformers`.
2. Inspect **Grouped-Query Attention (GQA)** KV-cache compression ratios across Phi variants.
3. Benchmark step-by-step chain-of-thought reasoning prompts on a Google Colab Free Tier T4 GPU.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes torch

## 1. Inspect Phi-4-mini Architecture & GQA KV-Cache Savings
Compare standard Multi-Head Attention (MHA) against Grouped-Query Attention (GQA) used in `Phi-4-mini-instruct` (24 query heads, 8 KV heads).

In [ ]:
def gqa_kv_cache_mb(layers: int, kv_heads: int, head_dim: int, seq_len: int, bytes_per_elem: int = 2) -> float:
    total_bytes = 2 * layers * kv_heads * head_dim * seq_len * bytes_per_elem
    return total_bytes / (1024 ** 2)

mha_mb = gqa_kv_cache_mb(layers=32, kv_heads=24, head_dim=128, seq_len=8192)
gqa_mb = gqa_kv_cache_mb(layers=32, kv_heads=8, head_dim=128, seq_len=8192)
print(f"Standard MHA KV Cache (8K ctx): {mha_mb:.1f} MB")
print(f"Phi-4-mini GQA KV Cache (8K ctx): {gqa_mb:.1f} MB ({mha_mb / gqa_mb:.1f}x reduction)")

## 2. Run 4-Bit Inference on Colab T4 GPU
We load `microsoft/Phi-4-mini-instruct` in 4-bit `bitsandbytes` NF4 quantization when a GPU is available, or fall back to a tiny CPU-friendly checkpoint.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.cuda.is_available():
    from transformers import BitsAndBytesConfig
    model_id = "microsoft/Phi-4-mini-instruct"
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
    messages = [{"role": "user", "content": "Solve step by step: A factory edge node processes 45 frames/sec. How many frames in 2 minutes?"}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    out = model.generate(inputs, max_new_tokens=128, do_sample=False)
    print(tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))
else:
    print("CPU runtime detected — switch Colab Runtime -> Change runtime type -> T4 GPU to run 4-bit Phi-4-mini live.")